# 03- Implementación modelos de clasificación supervisada

**Proyecto Final Machine Learning**  
Adriana Lorenzo Mingorance

En este notebook vamos a construir distintos modelos de clasificación supervisada para predecir la variable objetivo: la situación final del estudiante (abandono, matriculado o graduado). 

Los modelos que vamos a implementar son:
1. Regresión logística
2. KNN
3. Random Forest 
4. Boosting


## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import cross_val_score

semilla = 42

## 1. Carga de datos

In [2]:
X_train = pd.read_csv('../data/X_train_clf.csv')
X_test = pd.read_csv('../data/X_test_clf.csv')

X_train_sc = pd.read_csv('../data/X_train_clf_scaled.csv')
X_test_sc = pd.read_csv('../data/X_test_clf_scaled.csv')

y_train = pd.read_csv('../data/y_train_clf.csv')['objetivo']
y_test = pd.read_csv('../data/y_test_clf.csv')['objetivo'] #para convertirlo a una serie (vector 1D)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Clases: {y_train.value_counts().to_dict()}")


Train: (3539, 237) | Test: (885, 237)
Clases: {'graduado': 1767, 'abandono': 1137, 'matriculado': 635}


## 2. Análisis del desbalanceo de clases

In [3]:
porcentajes = y_train.value_counts(normalize=True)*100
print("Distribución en train: ")
for clase, p in porcentajes.items():
    print(f" {clase}: {p:.1f}%")

#ratio de desbalanceo
ratio = porcentajes.max() / porcentajes.min()
print(f"\nRatio max/min: {ratio:.1f}x")

if ratio > 1.5:
    print("Hay cierto desbalanceo; tendremos esto en cuenta en la evaluación")

Distribución en train: 
 graduado: 49.9%
 abandono: 32.1%
 matriculado: 17.9%

Ratio max/min: 2.8x
Hay cierto desbalanceo; tendremos esto en cuenta en la evaluación


## 3. Función auxiliar de evaluación de modelos

In [4]:
resultados = {} #para guardar métricas de todos los modelos

def evaluar_modelo(nombre, modelo, X_tr, X_test, y_tr, y_test):
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_test)

    #usamos las metricas que hemos importdo de sklearn
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_pond = f1_score(y_test, y_pred, average='weighted')

    #hacemos validacion cruzada con k= 5 folds, sobre train
    cv_scores = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='f1_macro')

    #guardamos los resultados
    resultados[nombre] = {
        'accuracy': round(acc, 4),
        'F1 macro': round(f1_macro,4),
        'F1 ponderado': round(f1_pond, 4),
        'CV F1 macro (mean)': round(cv_scores.mean(), 4),
        'CV F1 macro (std)': round(cv_scores.std(), 4)
    }

    #imprimimos todo:
    print(f"\n{nombre}: ")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 macro: {f1_macro:.4f}")
    print(f"F1 ponderado: {f1_pond:.4f}")
    print(f"CV F1 macro: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
    print()
    print(classification_report(y_test, y_pred))

    return modelo, y_pred

## 4. Modelo 1: Regresión logística

In [5]:
reg_log = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=semilla)

#usamos los csv escalados para evitar sesos si las variables tienen escalas diferentes
reg_log_fit, reg_log_pred = evaluar_modelo('Regresión Logística', reg_log, X_train_sc, X_test_sc, y_train, y_test)


Regresión Logística: 
Accuracy: 0.7345
F1 macro: 0.6862
F1 ponderado: 0.7436
CV F1 macro: 0.7011 +/- 0.0080

              precision    recall  f1-score   support

    abandono       0.80      0.71      0.75       284
    graduado       0.86      0.82      0.84       442
 matriculado       0.41      0.55      0.47       159

    accuracy                           0.73       885
   macro avg       0.69      0.69      0.69       885
weighted avg       0.76      0.73      0.74       885



La regresión logísitica proporciona una baseline sólido, con un buen rendimienglo gloal (vemos F1 macro = 0.69 aprox). Sin embargo, vemos que la clase matriculado es algo más difícil de predecir, lo que sugiere una menor separabilidad en el espacio de caracterísitcas respecto a las clases 'abandono' y 'graduado'

## 5. Modelo 2: KNN

In [17]:
"""#aseguramos que este en float para KNN
X_train_sc = X_train_sc.astype(float)
X_test_sc = X_test_sc.astype(float)

#hacemos primero una busqueda pequeña y simple de k, según la practica 2
k_valores = [3,5,7,9,11]
best_k = None
best_score = -1

print("Búsqueda simple de k: ")
for k in k_valores:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_sc, y_train, cv=5, scoring='f1_macro')
    mean_score = scores.mean()
    print(f"K = {k}: CV F1 macro={mean_score:.4f}")

    #cogemos el mejor, vamos actualizando
    if mean_score > best_score:
        best_score = mean_score
        best_k = k

print(f"\nMejor k = {best_k}")
"""


'#aseguramos que este en float para KNN\nX_train_sc = X_train_sc.astype(float)\nX_test_sc = X_test_sc.astype(float)\n\n#hacemos primero una busqueda pequeña y simple de k, según la practica 2\nk_valores = [3,5,7,9,11]\nbest_k = None\nbest_score = -1\n\nprint("Búsqueda simple de k: ")\nfor k in k_valores:\n    knn = KNeighborsClassifier(n_neighbors=k)\n    scores = cross_val_score(knn, X_train_sc, y_train, cv=5, scoring=\'f1_macro\')\n    mean_score = scores.mean()\n    print(f"K = {k}: CV F1 macro={mean_score:.4f}")\n\n    #cogemos el mejor, vamos actualizando\n    if mean_score > best_score:\n        best_score = mean_score\n        best_k = k\n\nprint(f"\nMejor k = {best_k}")\n'

## 6. Modelo 3: Random Forest

In [6]:
random_forest = RandomForestClassifier(n_estimators=200, max_depth=None, class_weight='balanced', random_state=semilla, n_jobs=-1)

#árboles no necesitan escalado, usamos los datasets normales
rf_fit, rf_pred = evaluar_modelo('Random Forest', random_forest, X_train, X_test, y_train, y_test)


Random Forest: 
Accuracy: 0.7729
F1 macro: 0.6853
F1 ponderado: 0.7524
CV F1 macro: 0.6766 +/- 0.0138

              precision    recall  f1-score   support

    abandono       0.81      0.74      0.77       284
    graduado       0.77      0.96      0.86       442
 matriculado       0.64      0.32      0.43       159

    accuracy                           0.77       885
   macro avg       0.74      0.67      0.69       885
weighted avg       0.76      0.77      0.75       885



Ahora vemos que Random Forest mejora ligeramente el rendimiento global respecto a regresión logísitca, sobre todo en accuracy y en la detección de 'graduado' (vemos su recall muy alto: 0.96). No obstante, seguimos viendo dificultades para identificar 'matriculado', lo que nos confirma que es la clase más compleja.

## 7. Modelo 4: Gradient Boosting

In [7]:
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=semilla)

gb_fit, gb_pred = evaluar_modelo('Gradient Boosting', gb, X_train, X_test, y_train, y_test)



Gradient Boosting: 
Accuracy: 0.7638
F1 macro: 0.6901
F1 ponderado: 0.7549
CV F1 macro: 0.7078 +/- 0.0091

              precision    recall  f1-score   support

    abandono       0.82      0.72      0.77       284
    graduado       0.80      0.92      0.86       442
 matriculado       0.50      0.40      0.44       159

    accuracy                           0.76       885
   macro avg       0.71      0.68      0.69       885
weighted avg       0.75      0.76      0.75       885



Por último, en boosting, sabemos que ahora no hacemos muchos árboles independientes, sino que cada árbol va intentando arreglar los errores del árbol anterior. Vemos cómo obtiene el mejor F1 macro, lo que indica un mejor equilibrio entre clases. Además, identifica algo mejor la clase de 'matriculado'. Puede considerarse el modelo más adecuado, vamos a analizarlos todos en conjunto.

## 7. Comparación de modelos

In [8]:
df_resultados = pd.DataFrame(resultados).T
print("Comparativa de modelos: ")
print(df_resultados.to_string())

#identificamos el mejor segñun F1 macro
mejor = df_resultados['F1 macro'].idxmax()
print(f"El mejor modelo es: {mejor}, con un F1 macro = {df_resultados.loc[mejor, 'F1 macro']}")

Comparativa de modelos: 
                     accuracy  F1 macro  F1 ponderado  CV F1 macro (mean)  CV F1 macro (std)
Regresión Logística    0.7345    0.6862        0.7436              0.7011             0.0080
Random Forest          0.7729    0.6853        0.7524              0.6766             0.0138
Gradient Boosting      0.7638    0.6901        0.7549              0.7078             0.0091
El mejor modelo es: Gradient Boosting, con un F1 macro = 0.6901
